# Лабораторная работа 5. Сервис синтеза

Юдин Артём М4221

**Цель работы:** собрать нормализатор, предиктор пауз, транскриптор и акустическую модель в один тракт: на входе текст, на выходе звук.

Ноутбук считает копи-синтез через HiFi-GAN, синтезирует 10 фраз из `data/evaluation_text.csv` и печатает время по этапам и RTF. Запускать его нужно там, где уже лежат веса FastSpeech2 и корпус RUSLAN: чекпоинты в git не входят. Каталог запуска — `labs/lab4_acoustic/runs/baseline/baseline_g2p`. Это модель, обученная на фонемах того же G2P. В лабораторной работе №4 её средний MCD был 27,2 дБ против 62,5 дБ у постоянного мел-спектра. Запуск MFA даёт меньший MCD, но он учился на GT-фонемах и внешних длительностях, поэтому к этому тракту он не подходит.

`speechbrain` и `num2words` добавлены в `pyproject.toml`. Ставить их нужно в уже собранное окружение лабораторной работы №4, чтобы не подменять torch 2.5.1+cu121 и форк Coqui. Полный `uv sync` по новому lock-файлу разрешает более свежий torch, который speechbrain тянет за собой.


# 1. Схема тракта и стыковка работ №1–4

```
текст
  → normalize        TextNormalizer, лабораторная работа №1
  → tokenize         токены label_raw: слово и пунктуация до следующего слова
  → insert_pauses    PausePredictor.predict_durations, вставляет <SIL>
  → to_phones        G2P, <SIL> проходит без изменений
  → куски примерно по 24 токена
  → mel              FastSpeech2, baseline_g2p
  → vocode           HiFi-GAN LibriTTS 22 050 Гц
звук
```

Публичные методы работ №1–3 не менялись. Тракт только подаёт им те формы, которые они уже описывают.

- Нормализатор сам приводит текст к NFC.
- Токен для предиктора пауз — это слово плюс пунктуация до следующего слова. Знаки до первого слова остаются на первом токене. Так устроен столбец `label_raw`.
- `G2P.transcribe` сам пропускает `<SIL>` и ищет слово в нижнем регистре без знака ударения и без пунктуации. Отдельно приводить ключ словаря в тракте не нужно.
- Словаря `russian_mfa.dict` в репозитории нет. Незнакомое слово идёт в буквенные правила G2P. Телефоны вне `common/phones_mfa.txt` отбрасываются.
- Модели загружаются один раз в `TTSPipeline.__init__`. Предиктор пауз при этом заново учится на всей таблице пауз, поэтому первая ячейка с инициализацией долгая. На каждый запрос модели не перезагружаются.

Длинная фраза режется уже в тракте, до FastSpeech2. Разрез ищет `<SIL>` или токен, который кончается знаком препинания, и оставляет паузу при предыдущем куске. Кусок не длиннее 25 токенов: лишний токен — это `<SIL>` сразу после разреза. Текст тестовых фраз не переписывался.

Числа нормализатор фактически не раскрывает. В работе №1 написано `import num2words`, а вызывается сам модуль. Это `TypeError`, его ловит `except`, и цифры остаются цифрами. Из токена `1800-х` транскриптор выбрасывает цифры и оставляет одну фонему `x`. Исправлять нормализатор я не стал: для стыковки хватает его текущего выхода, а дыра видна на `eval_09`.


In [ ]:
from pathlib import Path
import sys

import torch

HERE = Path.cwd().resolve()
ROOT = next(
    path for path in (HERE, *HERE.parents)
    if (path / "labs" / "lab5_service" / "tts_pipeline.py").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from labs.lab5_service.tts_pipeline import (  # noqa: E402
    SAMPLE_RATE,
    TTSPipeline,
    chunk_tokens,
    tokenize_text,
)

LAB5 = ROOT / "labs" / "lab5_service"
RUN_DIR = ROOT / "labs" / "lab4_acoustic" / "runs" / "baseline" / "baseline_g2p"
AUDIO_DIR = LAB5 / "audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device {device}, torch {torch.__version__}")
print(f"run {RUN_DIR}")
pipeline = TTSPipeline(RUN_DIR, device=device)


# 2. Вокодер

HiFi-GAN взят готовый: `speechbrain/tts-hifigan-libritts-22050Hz`. Он обучен не на настройках Coqui по умолчанию. Пять отличий, из-за которых копи-синтез либо звучит как запись, либо нет:

- `spec_gain` — 1, а не 20. Иначе мел в двадцать раз крупнее, чем ждёт вокодер.
- `log_func` — `np.log`, а не `np.log10`. Вокодер учился на натуральном логарифме.
- `signal_norm` — выключен. Лишняя min-max нормализация сдвигает диапазон.
- `do_trim_silence` — выключен. Обрезка тишины меняет число кадров.
- `mel_fmax` — 8000, а не `None`. Так указано в карточке модели.

Проверка ниже не использует акустическую модель. Настоящая запись RUSLAN превращается в мел этими настройками и сразу подаётся в HiFi-GAN. Рядом сохраняется Гриффин-Лим по тому же мелу: это потолок, который был слышен в работе №4. У настоящей записи RUSLAN мел лежит примерно в диапазоне [-10,2; 0,5]. Если диапазон другой, виновата аудио-конфигурация, а не FastSpeech2.

Ожидание на слух: `copysynth_hifigan.wav` близко к `natural.wav`, `copysynth_griffinlim.wav` заметно более звонкий и «металлический». Вокодер английский и женский по обучению, диктор RUSLAN — мужской, поэтому тембр всё равно чужой. Для копи-синтеза это не ошибка тракта: модели текста здесь нет.


In [ ]:
import torch
from IPython.display import Audio, Markdown, display
from TTS.config.shared_configs import BaseAudioConfig
from TTS.utils.audio import AudioProcessor

audio_config = BaseAudioConfig(
    sample_rate=22050,
    fft_size=1024,
    win_length=1024,
    hop_length=256,
    num_mels=80,
    mel_fmin=0.0,
    mel_fmax=8000.0,
    spec_gain=1.0,
    log_func="np.log",
    signal_norm=False,
    do_trim_silence=False,
    preemphasis=0.0,
    ref_level_db=20,
    trim_db=60.0,
)
ap = AudioProcessor.init_from_config(audio_config)

run_audio = pipeline.config.audio
for name, expected in (
    ("mel_fmax", 8000.0),
    ("spec_gain", 1.0),
    ("log_func", "np.log"),
    ("signal_norm", False),
    ("do_trim_silence", False),
):
    actual = getattr(run_audio, name)
    print(f"{name}: run={actual!r} vocoder={expected!r} match={actual == expected}")


def find_wav(root: Path) -> Path:
    directories = (
        root / "data" / "RUSLAN" / "RUSLAN",
        root / "data" / "ruslan_dataset" / "RUSLAN",
    )
    for directory in directories:
        preferred = directory / "000000_RUSLAN.wav"
        if preferred.is_file():
            return preferred
        if directory.is_dir():
            found = sorted(directory.glob("*.wav"))
            if found:
                return found[0]
    raise FileNotFoundError(
        "Нет wav в data/RUSLAN/RUSLAN или data/ruslan_dataset/RUSLAN"
    )


wav_path = find_wav(ROOT)
print(f"source {wav_path}")
natural = ap.load_wav(str(wav_path))
mel = ap.melspectrogram(natural)
print(f"mel {tuple(mel.shape)}, range [{mel.min():.2f}, {mel.max():.2f}]")
if mel.min() < -12 or mel.max() > 2:
    print("Диапазон далеко от [-10.2, 0.5]: конфигурация мела не совпадает с вокодером.")

mel_tensor = torch.tensor(mel, device=pipeline.device)
with torch.no_grad():
    hifigan = pipeline.vocoder.decode_batch(mel_tensor).squeeze().cpu().numpy()
griffin = ap.inv_melspectrogram(mel)

copies = {
    "natural.wav": natural,
    "copysynth_hifigan.wav": hifigan,
    "copysynth_griffinlim.wav": griffin,
}
for name, signal in copies.items():
    path = AUDIO_DIR / name
    ap.save_wav(signal, str(path), sr=22050)
    display(Markdown(f"**{name}**"))
    display(Audio(signal, rate=SAMPLE_RATE))


# 3. Десять фраз и RTF

Фразы берутся из `data/evaluation_text.csv` как есть. RTF — это время всего вызова, делённое на длительность полученного звука. RTF меньше 1 значит, что синтез быстрее реального времени. Время по этапам суммируется, если фраза пошла несколькими кусками.

Первый вызов после загрузки всё ещё может быть дороже следующих: прогрев CUDA попадает в первую фразу. Длинная `eval_09` должна занимать больше абсолютного времени. RTF при этом не обязан расти, если куски обрабатываются примерно с той же скоростью.


In [ ]:
import csv

import pandas as pd
from IPython.display import Audio, Markdown, display

eval_path = LAB5 / "data" / "evaluation_text.csv"
rows = list(csv.DictReader(eval_path.open(encoding="utf-8"), delimiter="|"))
records = []
for row in rows:
    utterance_id = row["id"]
    text = row["text"]
    normalized = pipeline.normalize(text)
    tokens = tokenize_text(normalized)
    groups = chunk_tokens(tokens)
    wav, timings = pipeline(text)
    pipeline.save(wav, AUDIO_DIR / f"{utterance_id}.wav")
    display(Markdown(f"### {utterance_id}"))
    display(Markdown(text))
    print(f"токенов {len(tokens)}, кусков {len(groups)}, нормализовано: {normalized}")
    if len(wav):
        display(Audio(wav, rate=SAMPLE_RATE))
    else:
        print("пустая волна")
    print(timings.report())
    record = {
        "id": utterance_id,
        "chars": len(text),
        "tokens": len(tokens),
        "chunks": len(groups),
        "audio_s": timings.audio_seconds,
        "total_s": timings.total,
        "rtf": timings.rtf,
    }
    record.update({f"{name}_s": value for name, value in timings.stages.items()})
    records.append(record)

TIMINGS = pd.DataFrame(records)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(TIMINGS["chars"], TIMINGS["total_s"])
for _, row in TIMINGS.iterrows():
    axes[0].annotate(row["id"], (row["chars"], row["total_s"]), fontsize=8)
axes[0].set_xlabel("Символов во входе")
axes[0].set_ylabel("Время синтеза, с")
axes[0].set_title("Время от длины")

axes[1].scatter(TIMINGS["chars"], TIMINGS["rtf"])
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
for _, row in TIMINGS.iterrows():
    axes[1].annotate(row["id"], (row["chars"], row["rtf"]), fontsize=8)
axes[1].set_xlabel("Символов во входе")
axes[1].set_ylabel("RTF")
axes[1].set_title("RTF от длины")
plt.tight_layout()
plt.show()

TIMINGS.round(3)


# 4. Что получается на тестовых фразах

Слушать нужно файлы `audio/eval_01.wav` … `audio/eval_10.wav`. Ниже — что тракт делает с текстом ещё до акустической модели. Эти числа получены тем же нормализатором, токенизатором и G2P, без подгонки фраз.

**eval_01.** Обычное предложение, 14 токенов, один кусок. Запятая висит на токене `все,`, точка на последнем слове, так что предиктор пауз их видит. Это самый спокойный пример: бытовая лексика.

**eval_02.** Девять токенов и три короткие реплики: `что?`, `что!`, `что.`. Восклицание и вопрос не отделены от слова, предиктор получает их как класс пунктуации. Фраза короткая, 22 фонемы. На слух здесь проверяется, появляются ли паузы между репликами, а не качество редких слов.

**eval_03.** Тринадцать токенов и 112 фонем: имена науатль не из словаря. `шиупоуалли` читается буквенными правилами (`ʂ ɨ ʊ p ə ʊ a ʎ ɪ`), `Тлакашипеуалицтли` — тоже. `Шипе-Тотека` остаётся одним токеном из-за дефиса. Ударение ставит предиктор по хвосту слова, а не по языку-источнику. Ждать правильного науатля не стоит. Ждать стоит связную, пусть и неверную, русскую читку без падения.

**eval_04.** `Бэтмен` и `компьютере` в словарь не попали, `супермена` попал. Заимствования, записанные кириллицей, не пропадают: для них есть правило. Ошибка будет в ударении и в мягкости, не в пропуске слова.

**eval_05.** `передохнут` в словаре нет. Оба вхождения, «тараканы передохнут» и «люди передохнут», получают одну и ту же цепочку `pʲ ɪ rʲ ɪ d̪ ɐ x n̪ u t̪`. Омограф «умрут» / «отдохнут» тракт не различает. Двойной дефис в `люди --` приклеен к слову и не является тире из разметки пауз, поэтому на этом месте предиктор легко не поставит `<SIL>`, хотя по смыслу здесь граница.

**eval_06.** Скороговорка, 10 токенов. `тупогуб` в словаре нет и читается правилами. Здесь проверяется, не слипаются ли похожие слоги. Текст не упрощался.

**eval_07.** `зальцбург` — OOV, правила дают `z̪ ɐ ʎ t̪s̪ b u r k`. Запятая после `Ах` и восклицание на `бы!` до предиктора доходят.

**eval_08.** `fabric` и `details` дают пустой список фонем: в буквенных правилах нет латиницы. Слова из звука исчезают, русская рамка остаётся: «смотря какой, смотря сколько в этом пиджаке». Это не правка текста, а дыра инвентаря. Чтобы английские вставки зазвучали, их нужно заранее переписать кириллицей либо научить G2P латинице. Сейчас сервис выбирает первое: молча пропускает нечитаемое слово и говорит остальное.

**eval_09.** 189 токенов и 11 кусков. Это толстовская фраза, и целиком в FastSpeech2 её лучше не отправлять: выравниватель на такой длине разъезжается. Куски режутся по пунктуации, склейка — это конкатенация волн, без дополнительной паузы на шве. Швы могут быть слышны. `1800-х` не раскрывается в «тысяча восьмисотых» из-за сломанного вызова `num2words`, и в звуке от года остаётся только `х`. Остальной текст идёт как есть, включая длинные перечисления с `ни`.

**eval_10.** Мусорная строка не роняет процесс. Скобки и звёздочки нормализатор выкидывает, `+` становится «плюс», `=` становится «равно». Слеш он тоже выкидывает, поэтому `ТВ/спазм` склеивается в `ТВспазм`. Двоеточие остаётся на токене. На выходе 25 токенов, 2 куска и 147 фонем: сервис читает то, что уцелело, а не исходную разметку.

Общее ограничение акустической модели то же, что в работе №4. Голос роботизированный, слова при удачном ударении различимы. HiFi-GAN убирает металлический призвук Гриффина-Лима, но не делает из baseline_g2p другого диктора. Менять тестовые фразы под это я не стал.


# 5. Входы, которые обработать нельзя

Пустая строка, строка из пробелов и строка без единого слова (`???`) не вызывают исключение. Тракт возвращает волну нулевой длины. Латинское слово без русской опоры (`fabric`) токенизируется, но фонем не даёт, и результат тоже пустой. Так сервис отвечает на запрос, из которого нечего синтезировать: тишина вместо падения.

Смешанный вход вроде `eval_08` и `eval_10` — другой случай. Он обрабатывается частично. Нечитаемые куски пропадают, остальное произносится.


In [ ]:
for sample in ("", "   ", "???", "fabric"):
    wav, timings = pipeline(sample)
    print(repr(sample), "samples", len(wav), "seconds", round(timings.audio_seconds, 3))
    print(timings.report() or "  этапов нет")
